# Plotting notebook
For testing different plots

## Data Imports


In [6]:
import plot_OM as plom
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import pandas as pd

import regionmask

In [87]:
default_BB = plom.Kuroshio_BB
proj_choices = plom.PROJECTION_CHOICES(default_BB)
proj = proj_choices['albers_ee']
mid_lon = proj.proj4_params['lon_0']

### download data

In [ ]:
ds = plom.fetch_ARGO_data(default_BB, start_date='2025-01-01', end_date='2026-01-01')
ds.to_zarr('./data/Kuroshio_2025')

## Plotting


In [8]:
ds = xr.open_zarr('./data/Kuroshio_2025')
ds

<xarray.Dataset> Size: 607MB
Dimensions:                   (N_PROF: 17780, N_LEVELS: 396)
Coordinates:
    LATITUDE                  (N_PROF) float64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
  * N_LEVELS                  (N_LEVELS) int64 3kB 0 1 2 3 4 ... 392 393 394 395
  * N_PROF                    (N_PROF) int64 142kB 16698 11985 ... 15342 17591
    LONGITUDE                 (N_PROF) float64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    TIME                      (N_PROF) datetime64[ns] 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
Data variables: (12/23)
    CONFIG_MISSION_NUMBER     (N_PROF) int64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    DATA_MODE                 (N_PROF) <U1 71kB dask.array<chunksize=(17780,), meta=np.ndarray>
    CYCLE_NUMBER              (N_PROF) int64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    DIRECTION                 (N_PROF) <U1 71kB dask.array<chunksize=(17780,), meta=np.ndarray>
    PLATFORM_NUMBER           (N_PROF) int64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    POSITION_QC               (N_PROF) int64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    ...                        ...
    TEMP_ADJUSTED             (N_PROF, N_LEVELS) float32 28MB dask.array<chunksize=(2223, 99), meta=np.ndarray>
    TEMP_ADJUSTED_QC          (N_PROF, N_LEVELS) int64 56MB dask.array<chunksize=(2223, 50), meta=np.ndarray>
    TEMP_ADJUSTED_ERROR       (N_PROF, N_LEVELS) float32 28MB dask.array<chunksize=(2223, 99), meta=np.ndarray>
    TIME_QC                   (N_PROF) int64 142kB dask.array<chunksize=(17780,), meta=np.ndarray>
    VERTICAL_SAMPLING_SCHEME  (N_PROF) <U205 15MB dask.array<chunksize=(556,), meta=np.ndarray>
    TEMP_QC                   (N_PROF, N_LEVELS) int64 56MB dask.array<chunksize=(2223, 50), meta=np.ndarray>
Attributes:
    DATA_ID:              ARGO
    DOI:                  http://doi.org/10.17882/42182
    Fetched_from:         erddap.ifremer.fr
    Fetched_by:           nookh
    Fetched_date:         2026/02/23
    Fetched_constraints:  [x=114.00/130.50; y=0.00/16.83; z=0.0/200.0; t=2025...
    Fetched_uri:          https://erddap.ifremer.fr/erddap/tabledap/ArgoFloat...
    Processing_history:   Transformed with 'point2profile'

In [17]:
ds['month'] = ds.TIME.dt.month

### plotting different time windows

going to plot first 7, 14, 30 days of may 

In [66]:
## time ranges
window_7d = (
    pd.Timestamp('2025-05-01'), pd.Timestamp('2025-05-08')
)
window_14d = (
    pd.Timestamp('2025-05-01'), pd.Timestamp('2025-05-15')
)
window_1m = (
    pd.Timestamp('2025-05-01'), pd.Timestamp('2025-06-01')
)

def select_timerange(ds, window):
    daily = ds.TIME.dt.round('1d').compute()
    mask = (daily >= window[0]) & (daily < window[1])

    return ds.where(mask, drop=True)

In [88]:
def plot_obj_map(ds, ax, threshold=0.3):

    mp, ergrid, xg, yg = plom.calculate_objective_map(ds)

    land = regionmask.defined_regions.natural_earth_v5_0_0.land_110
    mask_land = land.mask(xg, yg)

    mask = ergrid > threshold

    masked_mp = mp.copy()   
    masked_mp[mask] = np.nan

    plot_mp = np.where(np.isnan(mask_land), masked_mp.T, np.nan)

    im = ax.contourf(xg - mid_lon, yg, plot_mp)
    #axs[0].scatter(xg, yg, transform=ccrs.PlateCarree())
    plom.format_geoaxes(ax)

    return im

In [ ]:
mp, ergrid, xg, yg = plom.calculate_objective_map(ds)

fig, axs = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': proj})
axs = axs.flatten()

land = regionmask.defined_regions.natural_earth_v5_0_0.land_110
mask_land = land.mask(xg, yg)

threshold = 0.3
mask = ergrid > threshold

masked_mp = mp.copy()   
masked_mp[mask] = np.nan

plot_mp = np.where(np.isnan(mask_land), masked_mp.T, np.nan)

im = axs[0].contourf(xg - mid_lon, yg, plot_mp)
#axs[0].scatter(xg, yg, transform=ccrs.PlateCarree())
plom.format_geoaxes(axs[0])

